<a href="https://colab.research.google.com/github/fdx-hw/cosc-650/blob/main/week3_prompt_engineering_2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Week 3 (starter): Prompts as Engineering Artifacts

Live model calls, via Claude (Anthropic API). Set `ANTHROPIC_API_KEY` before running -- without it, `LIVE` is False and the model calls fall back to clearly-labeled fixtures so the harness still runs end to end.

Dependencies: `sentence-transformers` (local). For live calls: `pip install anthropic` and an Anthropic API key (console.anthropic.com).

In [1]:
!pip install -q anthropic

In [2]:
from google.colab import userdata
import os, json, pathlib

def claude_chat(messages, model='claude-haiku-4-5-20251001', max_tokens=200, **kw):
    """Claude via the Anthropic API. Returns text, or None if no key (API-BLOCKED).

    `messages` follows the same shape used throughout this notebook -- a list of
    {'role': ..., 'content': ...} dicts where one entry may have role 'system'.
    The Anthropic SDK takes the system prompt as its own `system` argument rather
    than as a message, so it's pulled out here before the call.
    """
    key = userdata.get('ANTHROPIC_API_KEY')
    if not key:
        return None
    from anthropic import Anthropic
    client = Anthropic(api_key=key)
    system = next((m['content'] for m in messages if m['role'] == 'system'), None)
    turns = [m for m in messages if m['role'] != 'system']
    resp = client.messages.create(model=model, max_tokens=max_tokens, system=system, messages=turns, **kw)
    return resp.content[0].text

LIVE = userdata.get('ANTHROPIC_API_KEY') is not None
print('live model calls:', LIVE, '(fixtures used when False)')

live model calls: True (fixtures used when False)


In [3]:
os.environ['HF_HOME'] = str((pathlib.Path('.') / '.hf_cache').resolve())
from sentence_transformers import SentenceTransformer, util
emb = SentenceTransformer('sentence-transformers/all-MiniLM-L6-v2')
def exact_match(a, b):
    return float(str(a).strip().lower() == str(b).strip().lower())
def semantic_sim(a, b):
    e = emb.encode([a, b], convert_to_tensor=True, normalize_embeddings=True)
    return round(float(util.cos_sim(e[0], e[1])), 3)
print('metrics ready (exact-match + semantic)')

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

metrics ready (exact-match + semantic)


## Part 1: Versioned prompts and a test suite
Each prompt version now lives in its own file under `prompts/` (`v1.txt`, `v2.txt`) instead of inline, per the instructions. Task: classify a support ticket. v2 adds one intent rule plus one supporting few-shot example.

Each file contains a system-style instruction (role, label set, output contract), three-to-four few-shot `ticket -> {category, rationale}` examples, and chain-of-thought scaffolding that asks for the reasoning inside one brief, visible `rationale` sentence rather than a hidden step-by-step block.

In [4]:
PROMPT_V1 = pathlib.Path('week-03/prompts/v1.txt').read_text()
PROMPT_V2 = pathlib.Path('week-03/prompts//v2.txt').read_text()

tests = [
  {'id':1,'ticket':'I was charged twice this month, refund the duplicate.','cat':'billing','why':'duplicate charge'},
  {'id':2,'ticket':'The app crashes when I tap export.','cat':'technical','why':'crash on a feature'},
  {'id':3,'ticket':'I want to change my email but the save button does nothing.','cat':'account','why':'update profile detail'},
  {'id':4,'ticket':'Tracking has not updated in four days.','cat':'shipping','why':'delivery tracking'},
  {'id':5,'ticket':'Love the new dashboard, great work.','cat':'account','why':'feedback, no request'},
  {'id':6,'ticket':'Password reset email never arrives.','cat':'account','why':'password reset'},
  {'id':7,'ticket':'I paid for express but the box came late and crushed.','cat':'shipping','why':'delivery problem, money is context'},
  {'id':8,'ticket':'Explain the tax line on my invoice.','cat':'billing','why':'invoice question'},
  {'id':9,'ticket':'CSV import drops non-English rows.','cat':'technical','why':'import bug'},
  {'id':10,'ticket':'Close my account and delete my data.','cat':'account','why':'account closure'},
]
print('prompt versions:', 2, '| test cases:', len(tests))

prompt versions: 2 | test cases: 10


In [5]:
# Labeled fixtures stand in for model output when LIVE is False. v2 fixes #7 but regresses #3.
FIX = {
  'v1': {1:('billing','duplicate charge'),2:('technical','crash on export'),3:('account','change email'),4:('shipping','tracking'),5:('account','praise'),6:('account','reset email'),7:('billing','mentions paying'),8:('billing','invoice charge'),9:('technical','import drops rows'),10:('account','close account')},
  'v2': {1:('billing','duplicate charge'),2:('technical','crash on export'),3:('technical','save button broken'),4:('shipping','tracking'),5:('account','praise'),6:('account','reset email'),7:('shipping','late damaged delivery'),8:('billing','invoice charge'),9:('technical','import drops rows'),10:('account','close account')},
}

def _parse_json_response(txt):
    """Strip a ```json ... ``` fence if present, then parse."""
    if txt is None:
        return {}
    t = txt.strip()
    if t.startswith('```'):
        t = t.split('```')[1]
        t = t[len('json'):].strip() if t.lower().startswith('json') else t.strip()
    return json.loads(t)

def run_case(version, prompt, t):
    if LIVE:
        txt = claude_chat([
            {'role':'system','content': prompt},
            {'role':'user','content': 'Ticket: ' + t['ticket']},
        ])
        try:
            d = _parse_json_response(txt); return d.get('category',''), d.get('rationale','')
        except Exception:
            return '', txt or ''
    return FIX[version][t['id']]

def score(version, prompt):
    rows = []
    for t in tests:
        cat, why = run_case(version, prompt, t)
        rows.append({'id':t['id'],'exact':exact_match(t['cat'],cat),'sem':semantic_sim(t['why'],why),'got':cat})
    acc = sum(r['exact'] for r in rows)/len(rows)
    return acc, rows

acc1, r1 = score('v1', PROMPT_V1)
acc2, r2 = score('v2', PROMPT_V2)
print(f'v1 exact-match {acc1:.0%}   v2 exact-match {acc2:.0%}')

v1 exact-match 90%   v2 exact-match 100%


In [6]:
for t in tests:
    a = next(r for r in r1 if r['id'] == t['id'])
    b = next(r for r in r2 if r['id'] == t['id'])
    print(f"#{t['id']:2d} expected={t['cat']:<10s} v1 got={a['got']:<10s} exact={a['exact']}  v2 got={b['got']:<10s} exact={b['exact']}")

# 1 expected=billing    v1 got=billing    exact=1.0  v2 got=billing    exact=1.0
# 2 expected=technical  v1 got=technical  exact=1.0  v2 got=technical  exact=1.0
# 3 expected=account    v1 got=technical  exact=0.0  v2 got=account    exact=1.0
# 4 expected=shipping   v1 got=shipping   exact=1.0  v2 got=shipping   exact=1.0
# 5 expected=account    v1 got=account    exact=1.0  v2 got=account    exact=1.0
# 6 expected=account    v1 got=account    exact=1.0  v2 got=account    exact=1.0
# 7 expected=shipping   v1 got=shipping   exact=1.0  v2 got=shipping   exact=1.0
# 8 expected=billing    v1 got=billing    exact=1.0  v2 got=billing    exact=1.0
# 9 expected=technical  v1 got=technical  exact=1.0  v2 got=technical  exact=1.0
#10 expected=account    v1 got=account    exact=1.0  v2 got=account    exact=1.0


## Part 3 and 4: the tradeoff and the failure
Show one case the edit improved and one it regressed. The regression is your required failure.

In [7]:
SEM_THRESHOLD = 0.1

for t in tests:
    row1 = next(r for r in r1 if r['id'] == t['id'])
    row2 = next(r for r in r2 if r['id'] == t['id'])
    e1, e2 = row1['exact'], row2['exact']
    s1, s2 = row1['sem'], row2['sem']

    if e1 != e2:
        verdict = 'IMPROVED' if e2 > e1 else 'REGRESSED'
        print(f"#{t['id']} expected {t['cat']!r}: v1 {'ok' if e1 else 'miss'} -> v2 {'ok' if e2 else 'miss'}  [{verdict}]")
    elif abs(s2 - s1) >= SEM_THRESHOLD:
        verdict = 'SEM IMPROVED' if s2 > s1 else 'SEM REGRESSED'
        print(f"#{t['id']} expected {t['cat']!r}: label unchanged ({'ok' if e1 else 'miss'}), sem {s1:.3f} -> {s2:.3f}  [{verdict}]")

#3 expected 'account': v1 miss -> v2 ok  [IMPROVED]
#10 expected 'account': label unchanged (ok), sem 0.699 -> 0.522  [SEM REGRESSED]


**Results.** v1 got 9/10 (90%), missing only case 3. v2 got 10/10 (100%) -- every case correct, including the one the edit was written for.

| case | version | got | exact | semantic-sim |
|---|---|---|---|---|
| #3 (email-change ticket, expected account) | v1 | technical | 0.0 | -- |
| #3 (email-change ticket, expected account) | v2 | account | 1.0 | -- |
| #10 (account-closure ticket, expected account) | v1 | account | 1.0 | 0.699 |
| #10 (account-closure ticket, expected account) | v2 | account | 1.0 | 0.522 |

(No sem for #3 since it's a label flip, so exact match already tells the whole story there.)

**Case 3 is the improved case.** v1 reads "I want to change my email but the save button does nothing" as `technical`. The broken button, not the email change, drives the label. The new rule tells the model that a malfunctioning control described alongside an account-change request is incidental, not the point. v2 correctly calls it `account`. This one actually learnt and got exercised by a real failure to fix it.

**Case 10 is the required failure case, and it's worth sitting with.** Both versions still get the label right (`account`, for "close my account and delete my data"), so exact-match shows nothing wrong. But the rationale's semantic-sim score drops from 0.699 to 0.522 under v2 worse than the case 3 fix is good, in relative terms.

Here's what makes this interesting rather than just annoying: the previous v2 (a completely different rule, about payment mentioned as shipping context) produced almost the identical regression on this exact same case, from 0.691 down to 0.526 last time, 0.699 down to 0.522 this time. Two unrelated edits, two unrelated rules, the same case drifts by almost the same amount. That's too consistent to be a coincidence from one-off sampling noise. It suggests the cause isn't what either rule says. It's more likely that adding a fourth few-shot example at all, regardless of its content, shifts how the model phrases case 10's rationale specifically, away from wording that matches "account closure."

**Why this one case is so sensitive.** I don't have a fully confirmed answer, but my best guess: "close my account and delete my data" is a short, blunt ticket with only one plausible label, so the model doesn't need to lean on the examples to get the category right -- but exactly because there's not much to disambiguate, small shifts in the prompt's overall style (introduced by whatever the fourth example happens to look like) show up more in how it explains an easy call, not what it answers.

**How I'd resolve this instead of continuing to trade errors.** Since the effect survived a full change of rule and example, tweaking the wording again probably won't fix it -- I already effectively ran that experiment by accident. The next real test would be a controlled ablation: run v1 plus just the new rule sentence with no new example, and separately v1 plus just the new example with no rule sentence, to see which piece (if either alone) reproduces the case 10 drift. That would tell me whether it's the extra example specifically or something about prompt length/structure in general. Practically, though, since this is a semantic-quality dip on an already-correctly-labeled case rather than a wrong answer, and the actual fix (case 3) is a clean, confirmed win with no exact-match cost anywhere, I'd accept this tradeoff for now and flag case 10 as a known, watched side effect rather than keep iterating on the wording. The label is still right, which is what the ticket-routing system actually depends on. I'd only keep chasing it if the semantic metric were feeding into something that also mattered operationally, like a customer-facing explanation.

## Part 5: Submit
Store the prompt versions as files, run the suite (set your key for real calls), and open a pull request with the metric numbers and a linked research note. Rubric: versioned prompts (15), structured prompt (20), test suite with two metrics (25), tradeoff with numbers (25), PR hygiene (15).